# SATEC · Random Forest & Gradient Boosting — entrenamiento y prueba

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenSntg/SATEC-Carrion/blob/main/notebooks/03_random_forest_train_test.ipynb)

**Alerta temprana de brotes de la enfermedad de Carrión.** Este cuaderno entrena
y evalúa un **Random Forest** y un **Gradient Boosting Histograma** (scikit-learn)
que predicen, por provincia y con 4 semanas de anticipación, si una zona entrará
en **estado de brote**. Validación **temporal estricta** (entrenamiento ≤ 2019,
prueba 2020–2024) y métricas adecuadas a **eventos raros**.

Repositorio: <https://github.com/StevenSntg/SATEC-Carrion>

## 1. Dependencias y datos

In [ ]:
!pip -q install pyarrow 2>/dev/null
import pandas as pd, numpy as np

In [ ]:
URL = "https://github.com/StevenSntg/SATEC-Carrion/raw/main/data/processed/dataset_enriched.parquet"
df = pd.read_parquet(URL)
print("Panel provincia-semana:", df.shape)
df[["departamento","provincia","anio","semana","casos","q3","brote"]].head()

## 2. Variables de entrada y partición temporal

In [ ]:
# Las 24 variables de entrada (autorregresivas de casos, estacionalidad ciclica,
# clima con rezagos/medias moviles y tasa por 100 000 hab.). Se EXCLUYEN las
# claves, el objetivo y el canal endemico (q1,q2,q3) para no filtrar la etiqueta.
FEATURE_COLS = [
    "casos", "casos_lag1", "casos_lag2", "casos_lag4",
    "roll_mean4", "roll_mean8", "sin_semana", "cos_semana",
    "prec", "temp", "hum",
    "prec_lag4", "prec_lag8", "prec_roll4", "prec_roll8",
    "temp_lag4", "temp_lag8", "temp_roll4", "temp_roll8",
    "hum_lag4", "hum_lag8", "hum_roll4", "hum_roll8",
    "tasa",
]

def feature_matrix(d):
    X = d[FEATURE_COLS].astype(float).fillna(0.0)
    y = d["brote"].astype(int)
    return X, y

In [ ]:
# Validacion TEMPORAL estricta: se entrena con <= 2019 y se prueba con 2020-2024.
# No se mezclan anios entre particiones (evita fuga de informacion temporal).
train = df[df["anio"] <= 2019].reset_index(drop=True)
test  = df[df["anio"] >  2019].reset_index(drop=True)
Xtr, ytr = feature_matrix(train)
Xte, yte = feature_matrix(test)
print("Entrenamiento:", Xtr.shape, "| brotes:", int(ytr.sum()))
print("Prueba       :", Xte.shape, "| brotes:", int(yte.sum()),
      f"({100*yte.mean():.1f}% de la clase positiva)")

## 3. Entrenamiento: Random Forest y Gradient Boosting

Ambos modelos usan `class_weight="balanced"` para compensar el fuerte desbalance de clases (la prevalencia de brotes cae del ~10 % en ≤ 2019 al ~0,3 % en 2020–2024 por la subnotificación pandémica).

In [ ]:
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=42, n_jobs=-1).fit(Xtr, ytr)
gb = HistGradientBoostingClassifier(max_iter=300,
          class_weight="balanced", random_state=42).fit(Xtr, ytr)

## 4. Prueba (test 2020–2024)

In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    brier_score_loss)

def evaluar(y_true, y_pred, y_score=None):
    """Metricas adecuadas a eventos raros (la prevalencia de brotes cae del
    ~10% (<=2019) al ~0.3% en 2020-2024 por la subnotificacion pandemica)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    m = {
        "Exactitud":     accuracy_score(y_true, y_pred),
        "Precision":     precision_score(y_true, y_pred, zero_division=0),
        "Sensibilidad":  recall_score(y_true, y_pred, zero_division=0),
        "F1":            f1_score(y_true, y_pred, zero_division=0),
        "VN": int(tn), "FP": int(fp), "FN": int(fn), "VP": int(tp),
    }
    if y_score is not None:
        m["AUC-ROC"] = roc_auc_score(y_true, y_score)
        m["AUC-PR"]  = average_precision_score(y_true, y_score)
        m["Brier"]   = brier_score_loss(y_true, y_score)
    return m

In [ ]:
for nombre, clf in [("Random Forest", rf), ("Gradient Boosting", gb)]:
    score = clf.predict_proba(Xte)[:, 1]
    pred  = clf.predict(Xte)
    m = evaluar(yte, pred, score)
    print(f"\n=== {nombre} ===")
    for k, v in m.items():
        print(f"  {k:12s}: {v:.3f}" if isinstance(v, float) else f"  {k:12s}: {v}")

## 5. Importancia de variables y matriz de confusión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion(y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ax.imshow(cm / cm.max(), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], ","), ha="center", va="center",
                    fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > cm.max()*0.5 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No brote", "Brote"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No brote", "Brote"])
    ax.set_xlabel("Prediccion"); ax.set_ylabel("Observado"); ax.set_title(titulo)
    plt.tight_layout(); plt.show()

In [ ]:
plot_confusion(yte, rf.predict(Xte), "Random Forest — prueba 2020–2024")

In [ ]:
plot_confusion(yte, gb.predict(Xte), "Gradient Boosting — prueba 2020–2024")

In [ ]:
# Importancia media de impureza (MDI) del Random Forest.
import matplotlib.pyplot as plt
imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()
imp.tail(12).plot.barh(figsize=(7, 4.5), color="#009E73")
plt.xlabel("Importancia (MDI)"); plt.title("Random Forest — variables")
plt.tight_layout(); plt.show()

## 6. Comparación con el baseline (canal endémico)

In [ ]:
# Baseline epidemiologico clasico: el "canal endemico" de la OPS/MINSA marca
# brote si los casos de hoy superan el tercer cuartil historico (q3).
bp_pred = (test["casos"] > test["q3"]).astype(int).to_numpy()
print("Canal endemico (referencia):")
print({k: round(v, 3) if isinstance(v, float) else v
       for k, v in evaluar(yte, bp_pred).items()})

## 7. Validación de origen móvil (umbral óptimo)

Esquema **train ≤ Y−2 / val = Y−1 / test = Y** para cada año Y ∈ 2016–2024. El umbral se selecciona sobre la validación maximizando F1.

In [ ]:
from sklearn.metrics import fbeta_score, average_precision_score
import numpy as np
def best_t(yv, sv, beta=1.0):
    if int(np.sum(yv))==0: return 0.5
    cand=np.unique(np.concatenate([[0.0],np.sort(sv),[1.0]]))
    return float(max(cand, key=lambda t: fbeta_score(yv,(sv>=t).astype(int),beta=beta,zero_division=0)))
yt=[];yp=[];ys=[]
for Y in range(2016,2025):
    tr=df[df.anio<=Y-2]; va=df[df.anio==Y-1]; te=df[df.anio==Y]
    if len(tr)==0 or len(te)==0: continue
    Xtr2,ytr2=feature_matrix(tr); clf2=RandomForestClassifier(n_estimators=300,class_weight="balanced",random_state=42,n_jobs=-1).fit(Xtr2,ytr2)
    Xv,yv=feature_matrix(va); t=best_t(yv.to_numpy(),clf2.predict_proba(Xv)[:,1]) if len(va) else 0.5
    Xte2,yte2=feature_matrix(te); s=clf2.predict_proba(Xte2)[:,1]
    yt+=list(yte2);ys+=list(s);yp+=list((s>=t).astype(int))
print("Origen movil — RF:", evaluar(np.array(yt),np.array(yp),np.array(ys)))